### 1. 离线阶段 - 构建知识库

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter


# 加载文档
with open("./resources/评估.md", "r", encoding="utf-8") as f:
    md_content = f.read()
# 切分文档
    # 使用结构感知切分器
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "h1"),
            ("##", "h2"),
            ("###", "h3")
        ],
        strip_headers=True
    )
    # 进行切分
    docs: list[Document] = splitter.split_text(md_content)
# 准备向量模型
embaddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",
    dimensions=1024
)
# 准备向量数据库
vector_store = InMemoryVectorStore(embedding=embaddings)
# 调用存储方法
vector_store.add_documents(docs)

### 2. 在线阶段 - 实现Agentic RAG形式的智能体

In [ ]:
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from app.core.config import settings
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 1. 准备模型
model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={
        "thinking":{
            "type": "disabled"
        }
    }
)
# 2. 准备工具
@tool
def rag_search(query: str, k: int = 4) -> str:
    """
    根据问题， 去向量数据库中检索对应的只是片段
    :param query:用户的问题
    :param k:需要检索的片段数
    :return:得到知识片段
    """
    print("检索工具被调用~")
    # 2.1 检索
    doc_list: list[tuple[Document, float]] = vector_store.similarity_search_with_score(
        query=query, k=k
    )
    # 2.2 增强
    response = ""
    for doc, score in doc_list:
        print(f"分数= {score}")
        print(f"正文： {doc.page_content}")
        response += doc.page_content + "\n\n"
    return response

# 3. 准备系统提示词
system_prompt = """
你是一个教资考试专家，你需要根据提供的上下文回答用户的问题，
不能自己编造答案。答案要简短，直接引用原文。
"""
# 4. 准备Checkpoint
checkpoint = InMemorySaver()
# 5. 定义智能体
_config = {
    "configurable":{
        "thread_id": "1"
    }
}

agent = create_agent(
    model=model,
    tools=[rag_search],
    system_prompt=system_prompt,
    checkpointer=checkpoint
)
# 6. 调用智能体
agent_response = agent.stream_events({
    "messages": [
        HumanMessage(content="教育的目的是什么")
        ]
    },
    version="v3",
    config=_config
)

for msg in agent_response.messages:
    for text in msg.text:
        print(text, end="")